# LSC5 — Inferencia final T10 v2
**Ensamble Transformer + CNN1D + Lineal por 4 tramos · semillas 42, 7 y 123.**

Este notebook carga los nueve checkpoints definitivos de la corrida `a8c94d8c1fad`, procesa los 375 landmarks oficiales y genera `submission.csv`. No entrena ni ajusta los modelos y no requiere datos de entrenamiento.

**Kaggle:** añade los tres inputs privados (artefactos, landmarks oficiales y plantilla). Comprueba las rutas de la primera celda y ejecuta **Run All** cuando estés listo para generar la entrega definitiva. CPU es suficiente.

## 1. Configuración y dependencias
Las tres rutas de entrada son: ARTIFACT_PATH, TEST_LANDMARKS_DIR, SUBMISSION_TEMPLATE_PATH  . `ARTIFACT_PATH` puede apuntar al ZIP original, a la carpeta descomprimida o a la carpeta de Kaggle que lo contiene. Los hiperparámetros y la versión de preprocesamiento están fijados para mantener la compatibilidad exacta con los pesos.

In [ ]:
from pathlib import Path
import hashlib, json, zipfile, io, re
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# ===== Única sección de configuración de usuario =====
# Carpeta de Kaggle que contiene el ZIP original o los nueve checkpoints y su configuración.
ARTIFACT_PATH = Path('/kaggle/input/datasets/tu-usuario/archivo.zip-de-lsc5-t10-ens3-a8c94d8c1fad-artifacts')  # Ruta de los pesos
TEST_LANDMARKS_DIR = Path('/kaggle/input/datasets/tu-usuario/archivo-de-test-lsc5/landmarks')  # Landmarks oficiales a predecir
SUBMISSION_TEMPLATE_PATH = Path('/kaggle/input/datasets/tu-usuario/archivo-de-template-lsc5/submission_template.csv')  # Plantilla oficial de entrega
DEVICE_NAME = 'cpu'  # cpu recomendado; cuda opcional
CPU_THREADS = 4
OUT_DIR = Path('/kaggle/working/lsc5_t10_v2_inferencia') if Path('/kaggle/working').exists() else Path('./lsc5_t10_v2_inferencia')

# ===== Configuración congelada (NO editar) =====
PREPROCESS_MODE = 'mask_aware'
PREPROCESS_VERSION = 't10_trim2d_ens3_v2'
MAX_INTERPOLATION_GAP = 3
SEQ_LEN = 32
N_SEGMENTS = 4
SEEDS = [42, 7, 123]
D_MODEL, N_LAYERS, N_HEADS, FFN_DIM = 64, 2, 4, 128
HAND_DIM, BODY_DIM, FACE_DIM = 40, 20, 4
DROPOUT = 0.3
LIN_DROPOUT = 0.3
MIRROR_TTA = True
CLASS_NAMES = [f'sign_{i:02d}' for i in range(1, 6)]
ID_TO_CLASS = dict(enumerate(CLASS_NAMES))
EXPECTED_TEST_COUNT = 375
EXPECTED_TAG = 'a8c94d8c1fad'
ARTIFACT_ZIP_NAME = 'lsc5_t10_ens3_a8c94d8c1fad_artifacts.zip'
DEVICE = torch.device(DEVICE_NAME)
if DEVICE.type == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA no disponible; usa DEVICE_NAME = "cpu".')
if DEVICE.type == 'cpu':
    torch.set_num_threads(CPU_THREADS)
print('PyTorch:', torch.__version__, '| Dispositivo:', DEVICE)
print('Salida:', OUT_DIR / 'submission.csv')

## 2. Preprocesamiento de landmarks
Cada muestra se transforma individualmente a un tensor de 32 fotogramas y 436 características. No se calculan estadísticas globales sobre el conjunto oficial. Se conservan exactamente las operaciones de la corrida original.

### 2.1 Lectura y representación geométrica
Se separan los 1662 valores de MediaPipe Holistic en cuerpo, rostro y manos. La posición se expresa respecto a las muñecas y el torso y se escala por el ancho de hombros de esa misma muestra. Se calculan también diferencias temporales y se preservan las máscaras de presencia.

In [ ]:
# Rebanadas de la representación de MediaPipe Holistic (1662 features/frame)
POSE_SLICE = slice(0, 132)
FACE_SLICE = slice(132, 1536)
LEFT_HAND_SLICE = slice(1536, 1599)
RIGHT_HAND_SLICE = slice(1599, 1662)

BODY_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24]  # hombros, codos, muñecas y caderas
FACE_JOINTS = [1, 33, 133, 362, 263, 61, 291, 13, 14, 70, 300]
BASE_FEATURE_DIM = 200
FEATURE_DIM = BASE_FEATURE_DIM * 2  # 400 (posición + velocidad)

# Índices anulados : profundidad Z y coordenadas locales del rostro
Z_IDX = list(range(2, 132, 3)) + list(range(134, 164, 4))
FACE_IDX = list(range(164, 197))
ZERO_COLS = Z_IDX + FACE_IDX + [i + 200 for i in Z_IDX + FACE_IDX]

def load_landmark_array(path):
    """Carga un .npy y valida la forma esperada (T, 1662) sin NaN/Inf."""
    arr = np.load(path, allow_pickle=False)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'{path}: esperaba matriz 2D, recibí {arr.shape}')
    if arr.shape[1] != 1662 and arr.shape[0] == 1662:
        arr = arr.T
    if arr.shape[1] != 1662:
        raise ValueError(f'{path}: esperaba 1662 características por frame, recibí {arr.shape}')
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if len(arr) < 2:
        raise ValueError(f'{path}: secuencia demasiado corta ({len(arr)} frames)')
    return arr


def landmarks_to_features(arr, target_len=SEQ_LEN, presence_override=None):
    """Construye (SEQ_LEN, 400) usando las máscaras de presencia observadas sobre video crudo."""
    t = len(arr)
    assert t == target_len

    pose = arr[:, POSE_SLICE].reshape(t, 33, 4)
    face = arr[:, FACE_SLICE].reshape(t, 468, 3)
    left_hand = arr[:, LEFT_HAND_SLICE].reshape(t, 21, 3)
    right_hand = arr[:, RIGHT_HAND_SLICE].reshape(t, 21, 3)

    # Escala corporal invariante a distancia: mediana del ancho de hombros
    left_shoulder = pose[:, 11, :3]
    right_shoulder = pose[:, 12, :3]
    torso_center = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder[:, :2] - right_shoulder[:, :2], axis=1)
    valid_widths = shoulder_width[shoulder_width > 1e-4]
    body_scale = max(float(np.median(valid_widths)) if len(valid_widths) else 1.0, 1e-3)

    def hand_features(hand, observed=None):
        geometric_present = (np.max(np.abs(hand), axis=(1, 2)) > 1e-6).astype(np.float32)
        present = geometric_present if observed is None else observed
        wrist = hand[:, 0, :]
        local = (hand - wrist[:, None, :]) / body_scale
        global_wrist = (wrist - torso_center) / body_scale
        local[geometric_present == 0] = 0.0
        global_wrist[geometric_present == 0] = 0.0
        return local.reshape(t, -1), global_wrist, present

    left_local, left_wrist, left_present = hand_features(left_hand, None if presence_override is None else presence_override[:, 0])
    right_local, right_wrist, right_present = hand_features(right_hand, None if presence_override is None else presence_override[:, 1])

    body_xyz = (pose[:, BODY_JOINTS, :3] - torso_center[:, None, :]) / body_scale
    body_visibility = pose[:, BODY_JOINTS, 3:4]
    body_features = np.concatenate([body_xyz, body_visibility], axis=2).reshape(t, -1)

    selected_face = face[:, FACE_JOINTS, :]
    nose = face[:, 1, :]
    eye_width = np.linalg.norm(face[:, 33, :2] - face[:, 263, :2], axis=1)
    valid_eye_widths = eye_width[eye_width > 1e-4]
    face_scale = max(float(np.median(valid_eye_widths)) if len(valid_eye_widths) else 1.0, 1e-3)
    face_present = (np.max(np.abs(face), axis=(1, 2)) > 1e-6).astype(np.float32)
    face_local = (selected_face - nose[:, None, :]) / face_scale
    face_local[face_present == 0] = 0.0
    if presence_override is not None:
        face_present = presence_override[:, 2]
    face_features = face_local.reshape(t, -1)

    base = np.concatenate([
        left_local, right_local,
        left_wrist, right_wrist,
        body_features, face_features,
        left_present[:, None], right_present[:, None], face_present[:, None]
    ], axis=1).astype(np.float32)

    velocity = np.diff(base, axis=0, prepend=base[:1]).astype(np.float32)
    velocity[:, -3:] = 0.0
    features = np.concatenate([base, velocity], axis=1).astype(np.float32)
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)

### 2.2 Espejo anatómico para TTA
Se refleja la coordenada horizontal y se intercambian las articulaciones simétricas y las manos. La versión original y la reflejada se evalúan con los mismos pesos y se promedian sus probabilidades; no se crean nuevos datos para ajustar el modelo.

In [ ]:
PAIRS = [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),(17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)]
FACE_PAIRS = [(33,263),(133,362),(61,291),(70,300)]


def _flipx(block):
    """Invierte x -> 1 - x solo en frames donde el bloque existe."""
    block = block.copy()
    present = np.abs(block).max(axis=2) > 1e-6
    block[..., 0] = np.where(present, 1.0 - block[..., 0], 0.0)
    return block


def mirror_raw(arr):
    """Genera la secuencia espejada intercambiando mano izquierda y derecha."""
    t = len(arr)
    pose = _flipx(arr[:, POSE_SLICE].reshape(t, 33, 4))
    face = _flipx(arr[:, FACE_SLICE].reshape(t, 468, 3))
    lh = _flipx(arr[:, LEFT_HAND_SLICE].reshape(t, 21, 3))
    rh = _flipx(arr[:, RIGHT_HAND_SLICE].reshape(t, 21, 3))
    for i, j in PAIRS:
        pose[:, [i, j]] = pose[:, [j, i]]
    for i, j in FACE_PAIRS:
        face[:, [i, j]] = face[:, [j, i]]
    return np.concatenate([pose.reshape(t, -1), face.reshape(t, -1),
                           rh.reshape(t, -1), lh.reshape(t, -1)], axis=1).astype(np.float32)

### 2.3 Características adicionales
Se añaden 30 ángulos de dedos calculados con las coordenadas originales y seis distancias 2D entre las muñecas y nariz, boca o pecho. Aunque se anulan las coordenadas faciales y Z directas, estas características derivadas se conservan tal como fueron usadas en la corrida original.

In [ ]:
EXTRA_DIM = 36
INPUT_DIM = FEATURE_DIM + EXTRA_DIM  # 436
FINGER_CHAINS = [(0, 1, 2, 3, 4), (0, 5, 6, 7, 8), (0, 9, 10, 11, 12), (0, 13, 14, 15, 16), (0, 17, 18, 19, 20)]


def _joint_angles(hand):
    """Calcula 15 ángulos internos por mano normalizados en [0, 1]."""
    out = []
    for chain in FINGER_CHAINS:
        for k in range(1, 4):
            p_prev, p_joint, p_next = hand[:, chain[k - 1]], hand[:, chain[k]], hand[:, chain[k + 1]]
            v1, v2 = p_prev - p_joint, p_next - p_joint
            denom = np.maximum(np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1), 1e-6)
            cos = np.clip(np.sum(v1 * v2, axis=1) / denom, -1.0, 1.0)
            out.append(np.arccos(cos) / np.pi)
    return np.stack(out, axis=1)


def extra_features(arr):
    """Extrae 36 rasgos geométricos extra: ang_izq(15), ang_der(15), dist_izq(3), dist_der(3)."""
    t = len(arr)
    pose = arr[:, POSE_SLICE].reshape(t, 33, 4)
    face = arr[:, FACE_SLICE].reshape(t, 468, 3)
    hands = [arr[:, LEFT_HAND_SLICE].reshape(t, 21, 3), arr[:, RIGHT_HAND_SLICE].reshape(t, 21, 3)]

    widths = np.linalg.norm(pose[:, 11, :2] - pose[:, 12, :2], axis=1)
    valid = widths[widths > 1e-4]
    scale = max(float(np.median(valid)) if len(valid) else 1.0, 1e-3)
    chest = pose[:, [11, 12, 23, 24], :2].mean(axis=1)
    face_present = np.max(np.abs(face), axis=(1, 2)) > 1e-6
    nose = face[:, 1, :2]
    mouth = (face[:, 13, :2] + face[:, 14, :2]) / 2.0

    blocks_ang, blocks_dist = [], []
    for hand in hands:
        present = np.max(np.abs(hand), axis=(1, 2)) > 1e-6
        ang = _joint_angles(hand)
        wrist = hand[:, 0, :2]
        dist = np.stack([np.linalg.norm(wrist - nose, axis=1),
                         np.linalg.norm(wrist - mouth, axis=1),
                         np.linalg.norm(wrist - chest, axis=1)], axis=1) / scale
        dist[~face_present, :2] = 0.0
        ang[~present] = 0.0
        dist[~present] = 0.0
        blocks_ang.append(ang)
        blocks_dist.append(dist)
    out = np.concatenate([blocks_ang[0], blocks_ang[1], blocks_dist[0], blocks_dist[1]], axis=1)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

### 2.4 Recorte activo, máscaras y remuestreo
El recorte localiza el tramo de movimiento mediante la elevación de las muñecas. El remuestreo `mask_aware` repara huecos interiores de hasta tres fotogramas sin alterar las máscaras de observación originales. Por último se construyen las 436 características y se anulan las columnas de profundidad y rostro directas y sus velocidades.

In [ ]:
def raw_presence_masks(raw):
    """Retorna 4 flags booleanos (pose, rostro, mano izq, mano der) por frame original."""
    t = len(raw)
    pose = raw[:, POSE_SLICE].reshape(t, 33, 4)
    face = raw[:, FACE_SLICE].reshape(t, 468, 3)
    lh = raw[:, LEFT_HAND_SLICE].reshape(t, 21, 3)
    rh = raw[:, RIGHT_HAND_SLICE].reshape(t, 21, 3)
    return np.stack([
        np.max(np.abs(pose[:, :, :3]), axis=(1, 2)) > 1e-6,
        np.max(np.abs(face), axis=(1, 2)) > 1e-6,
        np.max(np.abs(lh), axis=(1, 2)) > 1e-6,
        np.max(np.abs(rh), axis=(1, 2)) > 1e-6,
    ], axis=1)


def fill_short_missing_runs(block, observed, max_gap=MAX_INTERPOLATION_GAP):
    """Imputa solo huecos interiores de hasta max_gap frames entre observaciones válidas."""
    block = np.asarray(block, dtype=np.float32)
    observed = np.asarray(observed, dtype=bool)
    out = block.copy()
    usable = observed.copy()
    valid_indices = np.flatnonzero(observed)
    for a, b in zip(valid_indices[:-1], valid_indices[1:]):
        count = int(b - a - 1)
        if 0 < count <= max_gap:
            for i in range(a + 1, b):
                w = (i - a) / (b - a)
                out[i] = (1.0 - w) * out[a] + w * out[b]
                usable[i] = True
    return out, usable


def resample_masked_block(block, usable, positions):
    """Interpola pares utilizables evitando mezclar frames válidos con ceros."""
    lo = np.floor(positions).astype(np.int64)
    hi = np.minimum(lo + 1, len(block) - 1)
    alpha = (positions - lo).astype(np.float32)
    shape = (len(positions),) + (1,) * (block.ndim - 1)
    w = alpha.reshape(shape)
    mix = (1.0 - w) * block[lo] + w * block[hi]
    both = usable[lo] & usable[hi]
    choose_lo = usable[lo] & (~usable[hi]) & (alpha < 0.5)
    choose_hi = usable[hi] & (~usable[lo]) & (alpha >= 0.5)
    out = np.zeros_like(mix)
    out[both] = mix[both]
    out[choose_lo] = block[lo[choose_lo]]
    out[choose_hi] = block[hi[choose_hi]]
    return out.astype(np.float32)


def resample_landmarks_mask_aware(raw, target_len=SEQ_LEN):
    """Remuestrea la secuencia a SEQ_LEN frames conservando las máscaras observadas originales."""
    raw = np.asarray(raw, dtype=np.float32)
    observed = raw_presence_masks(raw)
    positions = np.linspace(0, len(raw) - 1, target_len, dtype=np.float32)
    nearest = np.clip(np.floor(positions + 0.5).astype(np.int64), 0, len(raw) - 1)

    specs = [(POSE_SLICE, 0), (FACE_SLICE, 1), (LEFT_HAND_SLICE, 2), (RIGHT_HAND_SLICE, 3)]
    result = np.zeros((target_len, 1662), dtype=np.float32)
    for sl, j in specs:
        repaired, usable = fill_short_missing_runs(raw[:, sl], observed[:, j])
        result[:, sl] = resample_masked_block(repaired, usable, positions)

    original_masks = observed[nearest][:, [2, 3, 1]].astype(np.float32)
    return result, original_masks


def trim_active_motion_pose(raw, rest_rise=0.3, min_active=3, margin=2):
    """Recorta frames inactivos al inicio y final según la elevación de muñecas normalizada por hombros."""
    t = len(raw)
    if t <= 10:
        return raw
    pose = raw[:, POSE_SLICE].reshape(t, 33, 4)
    ls, rs = pose[:, 11, :2], pose[:, 12, :2]
    both = (np.abs(ls).sum(axis=1) > 1e-6) & (np.abs(rs).sum(axis=1) > 1e-6)
    if not both.any():
        return raw
    width = max(float(np.linalg.norm(ls[both] - rs[both], axis=-1).mean()), 1e-3)
    wrists = pose[:, [15, 16], :2]
    w_ok = np.abs(wrists).sum(axis=2) > 1e-6
    if not w_ok.any():
        return raw
    wrists_y = np.where(w_ok, wrists[:, :, 1] / width, np.nan)
    rise = np.nanpercentile(wrists_y, 90, axis=0) - wrists_y
    active = (np.nan_to_num(rise, nan=0.0) > rest_rise).any(axis=1)
    sustained = np.convolve(active.astype(np.int32), np.ones(min_active, dtype=np.int32), mode='valid') == min_active
    starts = np.flatnonzero(sustained)
    if len(starts) == 0:
        return raw
    first = max(int(starts[0]) - margin, 0)
    last = min(int(starts[-1]) + min_active + margin, t)
    return raw[first:last] if (last - first) >= 6 else raw


def full_features(raw, seq_len=SEQ_LEN):
    """Pipeline por muestra: recorte activo, remuestreo mask_aware y anulación de columnas directas Z/rostro."""
    raw = trim_active_motion_pose(raw, rest_rise=0.3, min_active=3, margin=2)
    arr, original_masks = resample_landmarks_mask_aware(raw, seq_len)
    base = landmarks_to_features(arr, seq_len, presence_override=original_masks)
    out = np.concatenate([base, extra_features(arr)], axis=1)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    out[:, ZERO_COLS] = 0.0
    assert out.shape == (seq_len, INPUT_DIM) and np.isfinite(out).all()
    return out

## 3. Entradas y arquitecturas originales
Transformer y CNN1D reciben secuencias normalizadas con la media y desviación estándar guardadas en sus checkpoints. El modelo lineal recibe las medias de cuatro tramos temporales y la desviación estándar global de cada característica (2180 valores).

In [ ]:
def seq_to_segment_vector(X_seq, n_segments=N_SEGMENTS):
    """Convierte (N, T, D) en vector plano de n_segments medias por tramo + 1 desviación estándar global."""
    segs = np.array_split(X_seq, n_segments, axis=1)
    means = [s.mean(axis=1) for s in segs]
    std = X_seq.std(axis=1)
    return np.concatenate(means + [std], axis=1).astype(np.float32)

def apply_sequence_scaler(X_data, scaler):
    """Aplica únicamente el escalador guardado en el checkpoint."""
    return ((X_data - scaler['mean'][None, None, :]) / scaler['std'][None, None, :]).astype(np.float32)

### 3.1 Definición de las tres redes
Se reconstruyen las capas originales para poder cargar sus parámetros: Transformer con atención temporal, CNN1D con convoluciones residuales y modelo lineal por segmentos. La definición de las clases no implica entrenamiento; sus pesos se reemplazan con los checkpoints en el siguiente bloque.

In [ ]:
HAND_STREAM_IDX = list(range(0, 132)) + list(range(200, 332)) + [197, 198] + list(range(400, 436))
BODY_STREAM_IDX = list(range(132, 164)) + list(range(332, 364))
FACE_STREAM_IDX = list(range(164, 197)) + list(range(364, 397)) + [199]
_all_idx = HAND_STREAM_IDX + BODY_STREAM_IDX + FACE_STREAM_IDX
assert len(set(_all_idx)) == len(_all_idx) and max(_all_idx) < INPUT_DIM


def _branch(n_in, n_out):
    """Proyección de una rama anatómica: Linear -> LayerNorm -> GELU."""
    return nn.Sequential(nn.Linear(n_in, n_out), nn.LayerNorm(n_out), nn.GELU())


class LSC5_Transformer(nn.Module):
    """Transformer compacto de 3 ramas anatómicas (84.034 parámetros)."""
    def __init__(self, dropout=DROPOUT, num_classes=5):
        super().__init__()
        self.register_buffer('hand_idx', torch.tensor(HAND_STREAM_IDX, dtype=torch.long))
        self.register_buffer('body_idx', torch.tensor(BODY_STREAM_IDX, dtype=torch.long))
        self.register_buffer('face_idx', torch.tensor(FACE_STREAM_IDX, dtype=torch.long))
        self.hand_proj = _branch(len(HAND_STREAM_IDX), HAND_DIM)
        self.body_proj = _branch(len(BODY_STREAM_IDX), BODY_DIM)
        self.face_proj = _branch(len(FACE_STREAM_IDX), FACE_DIM)
        self.fusion_norm = nn.LayerNorm(D_MODEL)
        self.pos_embed = nn.Parameter(torch.zeros(1, SEQ_LEN, D_MODEL))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.embed_dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=FFN_DIM,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=N_LAYERS, norm=nn.LayerNorm(D_MODEL), enable_nested_tensor=False
        )
        self.attn_pool = nn.Linear(D_MODEL, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(D_MODEL * 2), nn.Dropout(dropout), nn.Linear(D_MODEL * 2, num_classes)
        )

    def forward(self, x):
        tokens = torch.cat([
            self.hand_proj(x.index_select(2, self.hand_idx)),
            self.body_proj(x.index_select(2, self.body_idx)),
            self.face_proj(x.index_select(2, self.face_idx))
        ], dim=-1)
        tokens = self.embed_dropout(self.fusion_norm(tokens) + self.pos_embed)
        features = self.encoder(tokens)
        weights = torch.softmax(self.attn_pool(features), dim=1)
        attended = torch.sum(features * weights, dim=1)
        return self.classifier(torch.cat([attended, features.mean(dim=1)], dim=1))


class LSC5_CNN1D(nn.Module):
    """CNN 1D compacta (38.213 parámetros) con convoluciones depthwise-separable residuales."""
    def __init__(self, in_dim=INPUT_DIM, hidden=64, dropout=DROPOUT, num_classes=5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_dim, hidden, kernel_size=1), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout)
        )
        self.block1 = nn.Sequential(
            nn.Conv1d(hidden, hidden, kernel_size=5, padding=2, groups=hidden),
            nn.Conv1d(hidden, hidden, kernel_size=1), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout)
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1, groups=hidden),
            nn.Conv1d(hidden, hidden, kernel_size=1), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout)
        )
        self.head = nn.Sequential(nn.LayerNorm(hidden * 2), nn.Dropout(dropout), nn.Linear(hidden * 2, num_classes))

    def forward(self, x):
        h = self.stem(x.transpose(1, 2))
        h = h + self.block1(h)
        h = h + self.block2(h)
        pooled = torch.cat([h.mean(dim=-1), h.amax(dim=-1)], dim=1)
        return self.head(pooled)


class LinearSegmentModel(nn.Module):
    """Modelo lineal por 4 tramos temporales + desviación estándar global (15.265 parámetros)."""
    def __init__(self, in_dim, num_classes=5, dropout=LIN_DROPOUT):
        super().__init__()
        self.norm = nn.BatchNorm1d(in_dim)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.fc(self.drop(self.norm(x)))

## 4. Lectura de los checkpoints
Se localiza una única fuente dentro de la ruta indicada. Se leen exclusivamente `experiment_config.json` y los nueve archivos `.pt`, verificando sus huellas SHA-256 para evitar mezclar pesos de otras corridas. El ZIP no necesita extraerse.

In [ ]:
EXPECTED_SHA256 = {'experiment_config.json': 'cd9fba6ae544726012c9a6292f32dd636f3053c4e89ab904e4380a6037e66f44', 'lsc5_cnn1d_final_seed123.pt': '5e471c811c6ec17282c41ebf99cbf9616f1b47dbd0c14095cb966c1ff80e5788', 'lsc5_cnn1d_final_seed42.pt': '3f1f76c662a3de5db156060d08712a868a725134fe25fd8b30b8936cd3f0b78e', 'lsc5_cnn1d_final_seed7.pt': 'da0f06446f0a0d79a257e4755ab5d50a45e9e68c47847928a7ba707a9273c940', 'lsc5_linear_final_seed123.pt': '0bc06e7b19942332839c7c48fbd9cdd2264f1e7cd95c92696530690849f11561', 'lsc5_linear_final_seed42.pt': 'e8edf6f8ccf5cbed3b58cf0ce002a963ccee87edc70199b1eb9c6dbb81601952', 'lsc5_linear_final_seed7.pt': 'ab7fbae0c1db9aff656daf510b76700d91eaa51cbe10e94ca8742d97e93ce7b9', 'lsc5_transformer_final_seed123.pt': '6bca1b2d5f7084579b70e4bb4806e251afe23968f7cab142fc4f9df661bb7e11', 'lsc5_transformer_final_seed42.pt': 'f6590dfff0bb0bb4bb6d5ce4ec41c174ee1d15b2f14e1494972f0cd508d6d65a', 'lsc5_transformer_final_seed7.pt': '4f2ff1e74ce0479fd7971d0b3f36a58c3a782b97f310baae9543cdb0c5c85665'}

def resolve_artifacts():
    """Localiza los artefactos originales dentro de la ruta configurada."""
    p = Path(ARTIFACT_PATH)
    if not p.exists():
        raise FileNotFoundError(f'No existe ARTIFACT_PATH: {p}')
    if p.is_file():
        return p
    # Kaggle puede montar el ZIP intacto o una carpeta con sus archivos.
    candidates = set(p.rglob(ARTIFACT_ZIP_NAME))
    for cfg_path in p.rglob('experiment_config.json'):
        if all((cfg_path.parent / name).is_file() for name in EXPECTED_SHA256):
            if hashlib.sha256(cfg_path.read_bytes()).hexdigest() == EXPECTED_SHA256['experiment_config.json']:
                candidates.add(cfg_path.parent)
    candidates = sorted(candidates)
    if len(candidates) != 1:
        raise RuntimeError(f'Se esperaba una fuente de pesos; encontradas {candidates}. Indica el ZIP o la carpeta exacta en ARTIFACT_PATH.')
    return candidates[0]


def read_verified_artifacts(path):
    path = Path(path)
    if path.is_dir():
        payloads = {name: (path/name).read_bytes() for name in EXPECTED_SHA256}
    elif path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            payloads = {}
            for name in EXPECTED_SHA256:
                entries = [x for x in z.namelist() if Path(x).name == name and not x.endswith('/')]
                if len(entries) != 1:
                    raise ValueError(f'ZIP: archivo faltante o duplicado: {name}')
                payloads[name] = z.read(entries[0])
    else:
        raise ValueError('ARTIFACT_PATH debe ser un ZIP o una carpeta descomprimida.')
    for name, blob in payloads.items():
        if hashlib.sha256(blob).hexdigest() != EXPECTED_SHA256[name]:
            raise ValueError(f'SHA-256 distinto: {name}. Usa los artefactos originales de a8c94d8c1fad.')
    cfg = json.loads(payloads['experiment_config.json'])
    expected = dict(config_tag=EXPECTED_TAG, preprocess_mode=PREPROCESS_MODE,
                    preprocess_version=PREPROCESS_VERSION, seq_len=SEQ_LEN,
                    max_interpolation_gap=MAX_INTERPOLATION_GAP, n_segments=N_SEGMENTS,
                    seeds=SEEDS, mirror_tta=True, n_train=1125, classes=CLASS_NAMES)
    for k, v in expected.items():
        if cfg.get(k) != v: raise ValueError(f'Configuración incompatible: {k}')
    print('Configuración y 9 pesos verificados por SHA-256.')
    return cfg, payloads

ARTIFACT_SOURCE = resolve_artifacts()
CONFIG, ARTIFACT_BYTES = read_verified_artifacts(ARTIFACT_SOURCE)
print('Fuente:', ARTIFACT_SOURCE)

### 4.1 Cargar pesos y escaladores
Se reconstruyen los nueve modelos, se cargan sus estados con `weights_only=True` y se comprueba su compatibilidad. Se deshabilitan los gradientes y se activa `eval()` para mantener Dropout desactivado y las estadísticas de BatchNorm fijas.

In [ ]:
def load_models(payloads):
    models = {k: [] for k in ('transformer', 'cnn1d', 'linear')}
    specs = {
        'transformer': (LSC5_Transformer, 84034, 'final_epochs', 24),
        'cnn1d': (LSC5_CNN1D, 38213, 'final_epochs', 17),
        'linear': (lambda: LinearSegmentModel(INPUT_DIM*(N_SEGMENTS+1), len(CLASS_NAMES)), 15265, 'lin_epochs', 50)
    }
    for kind, (factory, count, epoch_key, epoch_count) in specs.items():
        for seed in SEEDS:
            name = f'lsc5_{kind}_final_seed{seed}.pt'
            checkpoint = torch.load(io.BytesIO(payloads[name]), map_location='cpu', weights_only=True)
            if checkpoint['seed'] != seed or checkpoint['preprocess_version'] != PREPROCESS_VERSION or checkpoint[epoch_key] != epoch_count:
                raise ValueError(f'Metadatos incompatibles: {name}')
            model = factory()
            if sum(p.numel() for p in model.parameters()) != count:
                raise ValueError(f'Arquitectura incompatible: {name}')
            if not all(torch.isfinite(v).all().item() for v in checkpoint['model_state_dict'].values()):
                raise ValueError(f'Tensores no finitos: {name}')
            model.load_state_dict(checkpoint['model_state_dict'], strict=True)
            scaler = None
            if kind != 'linear':
                scaler = {'mean': checkpoint['scaler_mean'].numpy(), 'std': checkpoint['scaler_std'].numpy()}
                if any(v.shape != (INPUT_DIM,) or not np.isfinite(v).all() for v in scaler.values()) or np.any(scaler['std'] <= 0):
                    raise ValueError(f'Escalador incompatible: {name}')
            elif checkpoint['in_dim'] != INPUT_DIM * (N_SEGMENTS+1):
                raise ValueError(f'Dimensión lineal incompatible: {name}')
            model.to(DEVICE).eval().requires_grad_(False)
            models[kind].append((model, scaler))
            print(f'OK {name} | {epoch_count} épocas | {count:,} parámetros')
    return models

MODELS = load_models(ARTIFACT_BYTES)
assert sum(map(len, MODELS.values())) == 9

## 5. Predicción y ensamble
Cada modelo produce probabilidades de las cinco clases mediante `softmax`. Primero se promedia la predicción original y la espejada; después, las tres semillas de cada arquitectura. Finalmente se asigna un tercio del peso a cada arquitectura. Se preserva el orden de las operaciones originales. La clase final se obtiene con `argmax`, sin reglas posteriores de corrección.

In [ ]:
def predict_seq_model(model, scaler, X_raw, X_mir=None, batch_size=64):
    """Predice probabilidades promediando secuencia original y espejada (TTA)."""
    model.eval()
    def _run(X_in):
        X_s = apply_sequence_scaler(X_in, scaler)
        loader = DataLoader(TensorDataset(torch.from_numpy(X_s), torch.zeros(len(X_s), dtype=torch.int64)),
                            batch_size=batch_size, shuffle=False, num_workers=0)
        outs = []
        with torch.no_grad():
            for xb, _ in loader:
                logits = model(xb.to(DEVICE, non_blocking=True))
                outs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        return np.concatenate(outs, axis=0)
    p = _run(X_raw)
    if MIRROR_TTA and X_mir is not None:
        p = 0.5 * (p + _run(X_mir))
    return p

def predict_linear_segment(net, X_v, X_v_mir=None):
    net.eval()
    with torch.no_grad():
        p1 = torch.softmax(net(torch.from_numpy(X_v).to(DEVICE)), dim=1).cpu().numpy()
        if MIRROR_TTA and X_v_mir is not None:
            p2 = torch.softmax(net(torch.from_numpy(X_v_mir).to(DEVICE)), dim=1).cpu().numpy()
            return 0.5 * (p1 + p2)
    return p1

def validate_probs(probs, n):
    """Rechaza resultados no finitos o probabilidades inválidas antes de exportar."""
    if probs.shape != (n, len(CLASS_NAMES)) or not np.isfinite(probs).all():
        raise ValueError('Probabilidades inválidas (forma o valores no finitos).')
    if np.any(probs < 0) or not np.allclose(probs.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError('Probabilidades no normalizadas.')

@torch.inference_mode()
def predict_ensemble(X, Xm):
    """Fusiona las nueve redes con los pesos uniformes originales."""
    if X.ndim != 3 or X.shape[1:] != (SEQ_LEN, INPUT_DIM) or Xm.shape != X.shape:
        raise ValueError('Forma de características inválida.')
    Xv = seq_to_segment_vector(X, N_SEGMENTS)
    Xmv = seq_to_segment_vector(Xm, N_SEGMENTS)
    t_probs = [predict_seq_model(model, scaler, X, Xm) for model, scaler in MODELS['transformer']]
    c_probs = [predict_seq_model(model, scaler, X, Xm) for model, scaler in MODELS['cnn1d']]
    l_probs = [predict_linear_segment(model, Xv, Xmv) for model, _ in MODELS['linear']]
    probs = (np.mean(t_probs, axis=0) + np.mean(c_probs, axis=0) + np.mean(l_probs, axis=0)) / 3.0
    validate_probs(probs, len(X))
    return probs


def build_features(paths):
    """Preprocesa cada archivo y su espejo en el orden de la plantilla."""
    X, Xm = [], []
    for p in tqdm(paths, desc='Extracción individual de características'):
        raw = load_landmark_array(p)
        X.append(full_features(raw, SEQ_LEN))
        Xm.append(full_features(mirror_raw(raw), SEQ_LEN))
    return np.stack(X), np.stack(Xm)

## 6. Preparar y exportar las predicciones oficiales
Se exige que los 375 identificadores de los archivos `.npy` coincidan exactamente con la plantilla y se conserva su orden. El CSV de salida tiene únicamente las columnas `video_id` y `predicted_sign`. Las comprobaciones de formato son parte de la generación de la entrega y no calculan accuracy ni usan etiquetas reales.

In [ ]:
def prepare_test_paths(test_dir, template_path):
    """Vincula los archivos oficiales con los IDs y orden de la plantilla."""
    test_dir, template_path = Path(test_dir), Path(template_path)
    if not test_dir.is_dir(): raise FileNotFoundError(test_dir)
    if not template_path.is_file(): raise FileNotFoundError(template_path)
    template = pd.read_csv(template_path, dtype={'video_id': str}, keep_default_na=False)
    if 'video_id' not in template.columns: raise ValueError('Falta video_id en plantilla.')
    ids = template['video_id'].tolist()
    if len(ids) != EXPECTED_TEST_COUNT or len(set(ids)) != len(ids) or any(not re.fullmatch(r'sample_[A-Za-z0-9_-]+', v) for v in ids):
        raise ValueError('Plantilla inválida: se requieren 375 IDs anónimos únicos.')
    paths = sorted(test_dir.rglob('*.npy'))
    by_id = {p.stem: p for p in paths}
    if len(paths) != EXPECTED_TEST_COUNT or len(by_id) != len(paths) or set(by_id) != set(ids):
        raise ValueError('Los 375 landmarks deben coincidir exactamente con la plantilla, sin duplicados ni extras.')
    return ids, [by_id[v] for v in ids]


def write_submission(ids, probs, out_dir):
    """Exporta las etiquetas del argmax y valida el CSV antes de publicarlo."""
    validate_probs(probs, len(ids))
    if len(ids) != EXPECTED_TEST_COUNT or len(set(ids)) != len(ids):
        raise ValueError('Número de identificadores inválido.')
    sub = pd.DataFrame({'video_id': ids, 'predicted_sign': [ID_TO_CLASS[int(k)] for k in probs.argmax(axis=1)]})
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    dest = out_dir/'submission.csv'
    tmp = out_dir/'submission.csv.tmp'
    sub.to_csv(tmp, index=False, encoding='utf-8')
    check = pd.read_csv(tmp, dtype=str, keep_default_na=False)
    if list(check.columns) != ['video_id','predicted_sign'] or check['video_id'].tolist() != ids or not check['predicted_sign'].isin(CLASS_NAMES).all():
        raise RuntimeError('El CSV no pasó la comprobación final.')
    tmp.replace(dest)
    print('CSV generado:', dest, '| filas:', len(check), '| columnas:', list(check.columns))
    return dest

### 6.1 Ejecutar inferencia y generar `submission.csv`
Esta celda ejecuta la cadena completa con los pesos congelados. Al finalizar, descarga `submission.csv` desde el panel Output de Kaggle, en `/kaggle/working/lsc5_t10_v2_inferencia/`.

In [ ]:
# Leer los identificadores en el orden oficial y localizar sus landmarks.
ids, paths = prepare_test_paths(TEST_LANDMARKS_DIR, SUBMISSION_TEMPLATE_PATH)

# Extraer características de cada muestra y de su versión espejada.
Xt, Xmt = build_features(paths)

# Ejecutar los nueve modelos congelados y fusionar sus probabilidades.
probs = predict_ensemble(Xt, Xmt)

# Convertir las probabilidades en etiquetas y guardar el CSV de entrega.
submission_path = write_submission(ids, probs, OUT_DIR)